# 3b – LISA su griglia H3 (King County)

**SPEC 2026** – Econometria Spaziale (Python Lab)

Equivalente Python di `R/3_lisa_h3.R`

- Queen contiguity su poligoni H3
- Moran’s I globale su `log(price)`
- LISA e mappa cluster
- Moran scatterplot

In [ ]:
!pip install -q geopandas pyarrow libpysal esda mapclassify

In [ ]:
import geopandas as gpd
import numpy as np
import os
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from libpysal.weights import Queen, lag_spatial
from esda import Moran, Moran_Local
import warnings
warnings.filterwarnings('ignore')
np.random.seed(123)

BASE = "https://github.com/vincnardelli/spec/raw/main/data"
for f in ["tanzania.parquet", "kc_house.parquet", "kc_grid.parquet",
          "italian_provinces.parquet", "visium_hne_points.parquet"]:
    if not os.path.exists(f):
        !wget -q {BASE}/{f}

## 1) Caricamento dati

In [ ]:
kc = gpd.read_parquet("kc_grid.parquet")
kc["logprice"] = np.log(kc["price"])
kc.head()

## 2) Visualizzazione esplorativa

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
kc.plot(column="logprice", legend=True, cmap="YlOrRd",
        legend_kwds={"label": "log(price)"}, ax=ax)
ax.set_title("King County \u2013 log(price) su griglia H3")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 3) Pesi spaziali \u2013 Queen contiguity

In [ ]:
w = Queen.from_dataframe(kc, silence_warnings=True)
w.transform = "r"
print(w)

In [ ]:
centroids = kc.geometry.centroid
lines = []
for i in w.neighbors:
    for j in w.neighbors[i]:
        lines.append(LineString([centroids.iloc[i], centroids.iloc[j]]))
network = gpd.GeoDataFrame(geometry=lines, crs=kc.crs)

fig, ax = plt.subplots(figsize=(8, 8))
kc.plot(ax=ax, facecolor="#f0f0f0", edgecolor="grey", linewidth=0.15)
network.plot(ax=ax, color="#404040", linewidth=0.3, alpha=0.6)
ax.set_title("Rete di contiguit\u00e0 Queen (H3)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4) Moran\u2019s I globale

In [ ]:
y = kc["logprice"].values

mi = Moran(y, w, permutations=999)
print(f"Moran's I:       {mi.I:.4f}")
print(f"Expected I:      {mi.EI:.4f}")
print(f"p-value (norm):  {mi.p_norm:.4f}")
print(f"p-value (sim):   {mi.p_sim:.4f}")
print(f"z-score (norm):  {mi.z_norm:.4f}")

## 5) LISA (Local Moran\u2019s I)

In [ ]:
lisa = Moran_Local(y, w, permutations=999)

q_labels = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
kc["lisa_cluster"] = [
    q_labels[q] if p < 0.05 else "Not significant"
    for q, p in zip(lisa.q, lisa.p_sim)
]

z_y = (y - y.mean()) / y.std()
kc["z_y"] = z_y
kc["lag_z_y"] = lag_spatial(w, z_y)

In [ ]:
colors = {
    "High-High": "#B2182B", "Low-Low": "#2166AC",
    "High-Low": "#EF8A62",  "Low-High": "#67A9CF",
    "Not significant": "#d9d9d9"
}

fig, ax = plt.subplots(figsize=(8, 8))
kc.plot(color=kc["lisa_cluster"].map(colors), edgecolor="white",
        linewidth=0.1, ax=ax)
handles = [plt.Rectangle((0,0),1,1, facecolor=c) for c in colors.values()]
ax.legend(handles, colors.keys(), title="LISA cluster", loc="lower left")
ax.set_title("LISA Cluster Map \u2013 log(price)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 6) Moran scatterplot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.axhline(0, color="grey", lw=0.5)
ax.axvline(0, color="grey", lw=0.5)

for label, color in colors.items():
    mask = kc["lisa_cluster"] == label
    ax.scatter(kc.loc[mask, "z_y"], kc.loc[mask, "lag_z_y"],
               c=color, label=label, s=20, alpha=0.5, edgecolors="k", lw=0.2)

xr = np.array([-3, 3])
ax.plot(xr, mi.I * xr, color="grey", lw=1)

ax.set_xlabel("z(log price)")
ax.set_ylabel("Wz(log price)")
ax.set_title("Moran scatterplot")
ax.legend(title="LISA cluster", fontsize=8)
plt.tight_layout()
plt.show()